# Dependencies
You need: 
- CobraPy
- Pytorch
- CVXPYlayers
- Gurobi

There may be others (like jupyter, pandas, etc) but I trustt you can find that?


In [2]:
# import
import cobra as cb

# load model
model = cb.io.load_json_model("../Cobra_Models/GPR_model_1.json")

# reset all exchange bounds to 0 and 1000
for ex in model.exchanges:
    ex.lower_bound = 0
    ex.upper_bound = 1000

# media
media = {
    "EX_A(e)": -10,
    "EX_B(e)": -10,
}

# set media
for met, uptake in media.items():
    model.reactions.get_by_id(met).lower_bound = uptake

# optimize
solution = model.optimize()

# print solution
print("Objective value:", solution.objective_value)
print("Fluxes:")
for rxn_id, flux in solution.fluxes.items():
    print(f"{rxn_id}: {flux}")




Set parameter Username
Set parameter LicenseID to value 2773321
Academic license - for non-commercial use only - expires 2027-02-01
Objective value: 20.0
Fluxes:
EX_A(e): -10.0
EX_B(e): -10.0
T_A: 10.0
T_B: -10.0
T_Bio: 20.0
AtoPrebio(c): 10.0
BtoPrebio(c): 10.0
bio1(c): 20.0
EX_Bio(e): 20.0


# Simulating Gene Knockouts

In [9]:
import numpy as np
import cobra as cb

# load model
model = cb.io.load_json_model("../Cobra_Models/GPR_model_1.json")

# reset all exchange bounds to 0 and 1000
for ex in model.exchanges:
    ex.lower_bound = 0
    ex.upper_bound = 1000

# media
media = {
    "EX_A(e)": -10,
    "EX_B(e)": -10,
}  

# set media
for met, uptake in media.items():
    model.reactions.get_by_id(met).lower_bound = uptake

# gene knockout (removes ability for description in comment)
model.genes.get_by_id("G1").knock_out() # G1-> transport A into Cell
# model.genes.get_by_id("G2").knock_out() # G2 -> transport B into Cell
# model.genes.get_by_id("G3").knock_out() # G3 -> convert A to biomass
# model.genes.get_by_id("G4").knock_out() # G4 -> convert B to biomass

# optimize
solution = model.optimize()

# print solution
print("Objective value:", solution.objective_value)
print("Fluxes:")
for rxn_id, flux in solution.fluxes.items():
    print(f"{rxn_id}: {flux}")

Objective value: 10.0
Fluxes:
EX_A(e): 0.0
EX_B(e): -10.0
T_A: 0.0
T_B: -10.0
T_Bio: 10.0
AtoPrebio(c): 0.0
BtoPrebio(c): 10.0
bio1(c): 10.0
EX_Bio(e): 10.0


# convert Cobra Model to CVXPY model

In [10]:
# import
import cobra as cb
from cobra.util.array import create_stoichiometric_matrix
import cvxpy as cp
import numpy as np

# load model
model = cb.io.load_json_model("../Cobra_Models/GPR_model_1.json")

# reset all exchange bounds to 0 and 1000
for ex in model.exchanges:
    ex.lower_bound = 0
    ex.upper_bound = 1000

# media
media = {
    "EX_A(e)": -10,
    "EX_B(e)": -10
    }

# set media
for met, uptake in media.items():
    model.reactions.get_by_id(met).lower_bound = uptake

# get stoichiometric matrix
S = create_stoichiometric_matrix(model, array_type="dok")
num_reactions = S.shape[1]

# define flux variable
v = cp.Variable(num_reactions)

# define objective
objective = cp.Maximize(v[model.reactions.index("bio1(c)")])

# define constraints
constraints = [S @ v == 0]  # mass balance constraints
for i, rxn in enumerate(model.reactions):
    constraints.append(v[i] >= rxn.lower_bound)
    constraints.append(v[i] <= rxn.upper_bound)

# solve optimization problem
problem = cp.Problem(objective, constraints)
problem.solve("gurobi", verbose=True)

# print results
print("Optimal objective value:", problem.value)
print("Fluxes:")
for i, rxn in enumerate(model.reactions):
    print(f"{rxn.id}: {v.value[i]}")

(CVXPY) Apr 11 02:01:19 PM: Your problem has 9 variables, 25 constraints, and 0 parameters.
(CVXPY) Apr 11 02:01:19 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Apr 11 02:01:19 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Apr 11 02:01:19 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Apr 11 02:01:19 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Apr 11 02:01:19 PM: Compiling problem (target solver=GUROBI).
(CVXPY) Apr 11 02:01:19 PM: Reduction chain: FlipObjective -> CvxAttr2Constr -> Qp2SymbolicQp -> QpMatrixStuffing -> GUROBI
(CVXPY) Apr 11 02:01:19 PM: Applying reduction FlipObjective
(CVXPY) Apr 11 02:01:19 PM: Applying reduction CvxAttr2Constr
(CVXPY) Apr 11 02:01:19 PM: Applying reduction Qp2SymbolicQp
(CVXPY) Apr 11 02:01:19 PM: Applying reduction QpMatrixStuffing
(CVXPY) Apr 11 02:01:19 PM: Ap

                                     CVXPY                                     
                                     v1.7.5                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
Set parameter OutputFlag to value 1
Set parameter QCPDual to value 1
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Linux Mint 22.3")

CPU model: AMD Ryzen 9 5900HX with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
QCPDual  1

Optimize a m

Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+01, 1e+03]

Presolve removed 25 rows and 9 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0   -2.0000000e+01   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective -2.000000000e+01


(CVXPY) Apr 11 02:01:19 PM: Problem status: optimal
(CVXPY) Apr 11 02:01:19 PM: Optimal value: 2.000e+01
(CVXPY) Apr 11 02:01:19 PM: Compilation took 8.339e-03 seconds
(CVXPY) Apr 11 02:01:19 PM: Solver (including time spent in interface) took 1.113e-02 seconds


-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------
Optimal objective value: 20.0
Fluxes:
EX_A(e): -10.0
EX_B(e): -10.0
T_A: 10.0
T_B: -10.0
T_Bio: 20.0
AtoPrebio(c): 10.0
BtoPrebio(c): 10.0
bio1(c): 20.0
EX_Bio(e): 20.0


# create CVXPYlayers model from model

In [ ]:
# import
import cobra as cb
from cobra.util.array import create_stoichiometric_matrix
import cvxpy as cp
from cvxpylayers.torch import CvxpyLayer
import numpy as np
import torch

# load model
model = cb.io.load_json_model("../Cobra_Models/GPR_model_1.json")

# reset all exchange bounds to 0 and 1000
for ex in model.exchanges:
    ex.lower_bound = 0
    ex.upper_bound = 1000

# media
media = {
    "EX_A(e)": -10,
    "EX_B(e)": -10
    }

# set media
for met, uptake in media.items():
    model.reactions.get_by_id(met).lower_bound = uptake

# get stoichiometric matrix
S = create_stoichiometric_matrix(model, array_type="lil")
num_reactions = S.shape[1]

# define parameters
# S_tensor = torch.tensor(S.todense(), dtype=torch.float32)
# S_param = cp.Parameter(S.shape, name="S")
# S_param.value = S
lb_param = cp.Parameter(num_reactions, name="lb")
ub_param = cp.Parameter(num_reactions, name="ub")

# define flux variable
v = cp.Variable(num_reactions)

# define objective
objective = cp.Maximize(v[model.reactions.index("bio1(c)")])

# define constraints
constraints = [S @ v == 0, # mass balance constraints
               v >= lb_param, # lower bound constraints
               v <= ub_param]  # upper bound constraints

# create as cvpylayers problem
problem = cp.Problem(objective, constraints)
cvxpylayer = CvxpyLayer(problem, parameters=[lb_param, ub_param], variables=[v])

# solve optimization problem
lb_tensor = torch.tensor([rxn.lower_bound for rxn in model.reactions], dtype=torch.float32)
ub_tensor = torch.tensor([rxn.upper_bound for rxn in model.reactions], dtype=torch.float32)
solution, = cvxpylayer(lb_tensor, ub_tensor) 

# print results
print("Optimal objective value:", problem.value)
print("Fluxes:")
for i, rxn in enumerate(model.reactions):
    print(f"{rxn.id}: {solution[i].item()}")


Optimal objective value: None
Fluxes:
EX_A(e): -10.000000572606384
EX_B(e): -10.000000572605837
T_A: 10.000000573355099
T_B: -10.000000573354711
T_Bio: 20.00000114913147
AtoPrebio(c): 10.000000574016731
BtoPrebio(c): 10.000000574016356
bio1(c): 20.00000114874473
EX_Bio(e): 20.000001149324394
